# Projet 01 — Profils de fonctionnement des bornes fontaines

**Apprentissage non supervisé — K-Means & ACP**

**Mission :** Construire une typologie descriptive des bornes et distinguer intensité de service, disponibilité et attente.

**Unité étudiée :** une borne fontaine observée pendant une semaine.

---

In [ ]:
# ============================================================
# Imports et configuration globale
# ============================================================
import glob, os, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, silhouette_samples, adjusted_rand_score

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

RANDOM_STATE = 42
SEEDS = [42, 0, 1, 7, 123]

print('Configuration OK')

---
# Phase 1 — Diagnostic et préparation des données
---

## Question 01 — Définir les observations

**Que représente une ligne dans le fichier et quelle unité comparerons-nous ?**

Chaque ligne du fichier `mesures_brutes.csv` représente **une borne fontaine observée pendant une semaine**.

On distingue :
- **Clés et contexte** (à exclure des distances) : `id_observation`, `commune`, `date_releve`
- **Variables descriptives candidates** (pour le clustering) : `volume_m3`, `passages`, `attente_min`, `ouverture_h`, `pression_bar`, `interruption_h`, `stockage_m3`

L'unité d'analyse finale est la borne-semaine. Les clés servent à identifier et interpréter, pas à calculer des distances.

In [ ]:
# Q01 — Localisation et aperçu du fichier
data_files = glob.glob('../**/01_bornes_fontaines/donnees/mesures_brutes.csv', recursive=True)
if not data_files:
    data_files = glob.glob('**/01_bornes_fontaines/donnees/mesures_brutes.csv', recursive=True)
if not data_files:
    data_files = glob.glob('C:/Users/Michel ONDO/Documents/Project AI/**/mesures_brutes.csv', recursive=True)
print('Fichier trouvé :', data_files)
DATA_PATH = data_files[0]

## Question 02 — Importer sans perdre d'information

In [ ]:
# Q02 — Chargement brut en texte intégral
df_brut = pd.read_csv(DATA_PATH, sep=';', encoding='utf-8-sig',
                       dtype=str, keep_default_na=False)

# Copie intacte pour comparaison ultérieure
df_raw_backup = df_brut.copy()

print(f'Dimensions : {df_brut.shape[0]} lignes × {df_brut.shape[1]} colonnes')
print(f'\nColonnes : {list(df_brut.columns)}')
print()
df_brut.head(5)

In [ ]:
# Q02 — Types lus (tout est object/str)
print(df_brut.dtypes)
print(f'\nNombre de lignes : {len(df_brut)}')
print(f'Nombre de colonnes : {len(df_brut.columns)}')

**Commentaire Q02 :** Toutes les cellules ont été lues comme du texte (`dtype=str`) avec `keep_default_na=False` pour ne pas transformer automatiquement les valeurs manquantes. Le séparateur est `;` et l'encodage `utf-8-sig`. On conserve `df_raw_backup` intacte.

## Question 03 — Auditer les clés et les répétitions

In [ ]:
# Q03 — Doublons stricts (toutes colonnes identiques)
nb_doublons_stricts = df_brut.duplicated().sum()
print(f'Doublons stricts (toutes colonnes) : {nb_doublons_stricts}')

# Clés absentes ou vides
missing_vals = ['', 'NA', '?', 'non renseigne', 'NaN', 'nan']
cles_absentes = df_brut['id_observation'].apply(lambda x: x.strip() in missing_vals).sum()
print(f'Clés id_observation absentes ou vides : {cles_absentes}')

# Clés dupliquées
id_counts = df_brut['id_observation'].value_counts()
ids_dupliques = id_counts[id_counts > 1]
print(f'\nClés id_observation répétées : {len(ids_dupliques)}')

if len(ids_dupliques) > 0:
    print('\nExemples de clés répétées :')
    for dup_id in ids_dupliques.index[:5]:
        subset = df_brut[df_brut['id_observation'] == dup_id]
        print(f'\n--- {dup_id} ({len(subset)} occurrences) ---')
        display(subset)

In [ ]:
# Q03 — Identifier doublons stricts vs clés contradictoires
if len(ids_dupliques) > 0:
    contradictoires = []
    vrais_doublons = []
    for dup_id in ids_dupliques.index:
        subset = df_brut[df_brut['id_observation'] == dup_id]
        if subset.duplicated().any():
            vrais_doublons.append(dup_id)
        else:
            contradictoires.append(dup_id)
    print(f'Vrais doublons (lignes identiques) : {len(vrais_doublons)}')
    print(f'Clés contradictoires (même ID, contenu différent) : {len(contradictoires)}')
    if contradictoires:
        print('\nExemples contradictoires :')
        for c in contradictoires[:3]:
            display(df_brut[df_brut['id_observation'] == c])

**Commentaire Q03 :** On distingue les vrais doublons (lignes strictement identiques, à dé-dupliquer) des clés contradictoires (même identifiant mais contenus différents, à mettre en attente pour investigation).

## Question 04 — Repérer les manquants

In [ ]:
# Q04 — Détecter les 5 formes de valeurs manquantes
missing_forms = ['', 'NA', '?', 'non renseigne', 'NaN']

def count_missing(series):
    """Compte les différentes formes de valeurs manquantes."""
    counts = {}
    stripped = series.str.strip()
    for form in missing_forms:
        counts[form if form else '(vide)'] = (stripped == form).sum()
    counts['Total'] = sum(v for k, v in counts.items())
    return counts

# Tableau par colonne
missing_report = pd.DataFrame({col: count_missing(df_brut[col]) for col in df_brut.columns}).T
print('=== Fréquence des valeurs manquantes par colonne ===')
display(missing_report)

In [ ]:
# Q04 — Fréquence par commune
def is_missing(val):
    return val.strip() in missing_forms

numeric_cols = ['volume_m3', 'passages', 'attente_min', 'ouverture_h',
                'pression_bar', 'interruption_h', 'stockage_m3']

# Harmoniser commune d'abord pour le comptage
df_brut['commune_clean'] = df_brut['commune'].str.strip().str.title()

missing_by_commune = pd.DataFrame()
for col in numeric_cols:
    mask = df_brut[col].apply(is_missing)
    tmp = df_brut[mask].groupby('commune_clean').size()
    missing_by_commune[col] = tmp

missing_by_commune = missing_by_commune.fillna(0).astype(int)
print('=== Manquants par commune et par colonne ===')
display(missing_by_commune)

df_brut.drop(columns=['commune_clean'], inplace=True)

**Commentaire Q04 :** On identifie 5 écritures d'absence différentes : chaîne vide, `NA`, `?`, `non renseigne` et `NaN`. Ces formes doivent toutes être harmonisées avant toute conversion numérique. On distingue bien une absence d'un zéro observé (un zéro est une mesure valide).

## Question 05 — Convertir les nombres et les unités

In [ ]:
# Q05 — Conversion contrôlée des variables numériques
df = df_brut.copy()

# Journal de conversion
conversion_log = []

def clean_numeric(series, col_name):
    """Nettoie une colonne numérique :
    1. Remplace les formes d'absence par NaN
    2. Retire les suffixes d'unité
    3. Remplace la virgule décimale
    4. Convertit en float
    """
    result = series.copy()
    n_total = len(result)
    
    # Étape 1 : marquer les absences
    mask_missing = result.str.strip().isin(missing_forms)
    n_missing = mask_missing.sum()
    result[mask_missing] = np.nan
    
    # Étape 2 : retirer les suffixes d'unité (m3, h, min, bar, passages, etc.)
    mask_not_null = result.notna()
    suffix_pattern = r'\s*(m3|m³|h|min|bar|passages|mbps|Mbps)\s*$'
    n_suffix = result[mask_not_null].str.contains(suffix_pattern, regex=True, na=False).sum()
    result[mask_not_null] = result[mask_not_null].apply(
        lambda x: re.sub(suffix_pattern, '', str(x)).strip() if pd.notna(x) else x
    )
    
    # Étape 3 : remplacer la virgule décimale par un point
    n_comma = result[mask_not_null].str.contains(',', na=False).sum()
    result = result.apply(lambda x: str(x).replace(',', '.') if pd.notna(x) else x)
    
    # Étape 4 : conversion en float
    numeric_result = pd.to_numeric(result, errors='coerce')
    n_invalid = mask_not_null.sum() - n_missing - numeric_result[mask_not_null & ~mask_missing].notna().sum()
    n_converted = numeric_result.notna().sum()
    
    conversion_log.append({
        'colonne': col_name,
        'total': n_total,
        'absences_detectees': int(n_missing),
        'suffixes_retires': int(n_suffix),
        'virgules_remplacees': int(n_comma),
        'converties': int(n_converted),
        'non_interpretables': int(max(0, n_invalid))
    })
    
    return numeric_result

for col in numeric_cols:
    df[col] = clean_numeric(df[col], col)

df_conversion_log = pd.DataFrame(conversion_log)
print('=== Journal de conversion ===')
display(df_conversion_log)
print(f'\nTypes après conversion :')
print(df[numeric_cols].dtypes)

**Commentaire Q05 :** Chaque colonne numérique a été convertie en suivant un processus en 4 étapes : détection des absences, retrait des suffixes d'unité (regex), remplacement des virgules décimales, puis conversion en `float`. Le journal conserve la trace de chaque étape. Les valeurs originales sont conservées dans `df_raw_backup`.

## Question 06 — Vérifier les contrats et le contexte

In [ ]:
# Q06 — Vérification des plages techniques
plages = {
    'volume_m3': (0, 5000),
    'passages': (0, 30000),
    'attente_min': (0, 240),
    'ouverture_h': (0, 168),
    'pression_bar': (0, 15),
    'interruption_h': (0, 168),
    'stockage_m3': (0, 1000)
}

print('=== Valeurs hors plage technique ===')
cas_hors_plage = []
for col, (vmin, vmax) in plages.items():
    mask = df[col].notna() & ((df[col] < vmin) | (df[col] > vmax))
    n = mask.sum()
    if n > 0:
        print(f'\n{col} [{vmin}, {vmax}] : {n} valeur(s) hors plage')
        for idx in df[mask].index[:5]:
            row = df.loc[idx]
            cas_hors_plage.append({
                'index': idx,
                'id_observation': row['id_observation'],
                'colonne': col,
                'valeur': row[col],
                'plage': f'[{vmin}, {vmax}]',
                'argument': f'Valeur {row[col]} dépasse la plage admise'
            })
            print(f'  idx={idx} id={row["id_observation"]} -> {col}={row[col]}')

In [ ]:
# Q06 — Contrainte métier : interruption_h <= ouverture_h
mask_constraint = (
    df['interruption_h'].notna() & df['ouverture_h'].notna() &
    (df['interruption_h'] > df['ouverture_h'])
)
n_violation = mask_constraint.sum()
print(f'\n=== Contrainte interruption_h <= ouverture_h ===')
print(f'Violations : {n_violation}')
if n_violation > 0:
    print('\nExemples :')
    display(df[mask_constraint][['id_observation', 'interruption_h', 'ouverture_h']].head(10))
    for idx in df[mask_constraint].index[:5]:
        row = df.loc[idx]
        cas_hors_plage.append({
            'index': idx,
            'id_observation': row['id_observation'],
            'colonne': 'interruption_h vs ouverture_h',
            'valeur': f'{row["interruption_h"]} vs {row["ouverture_h"]}',
            'plage': 'interruption <= ouverture',
            'argument': 'Interruption dépasse la durée d\'ouverture'
        })

In [ ]:
# Q06 — Vérification des dates
print('=== Vérification des dates ===')
date_series = df['date_releve'].copy()

# Détecter les formats
mask_french = date_series.str.match(r'^\d{2}/\d{2}/\d{4}', na=False)
mask_iso = date_series.str.match(r'^\d{4}-\d{2}-\d{2}', na=False)
mask_other = ~mask_french & ~mask_iso & date_series.str.strip().apply(lambda x: x not in missing_forms)

print(f'Format ISO (AAAA-MM-JJ) : {mask_iso.sum()}')
print(f'Format français (JJ/MM/AAAA) : {mask_french.sum()}')
print(f'Autres / non interprétables : {mask_other.sum()}')

# Tenter la conversion
dates_parsed = pd.Series(index=df.index, dtype='datetime64[ns]')
dates_parsed[mask_iso] = pd.to_datetime(date_series[mask_iso], format='mixed', errors='coerce')
dates_parsed[mask_french] = pd.to_datetime(date_series[mask_french], format='%d/%m/%Y', errors='coerce')

n_dates_invalides = dates_parsed.isna().sum() - date_series.str.strip().isin(missing_forms).sum()
print(f'Dates non interprétables après parsing : {n_dates_invalides}')

# Dates hors de la plage attendue [2025-01-01, 2026-08-23]
if dates_parsed.notna().any():
    hors_plage_dates = dates_parsed[
        dates_parsed.notna() & 
        ((dates_parsed < '2025-01-01') | (dates_parsed > '2026-08-23'))
    ]
    print(f'Dates hors plage [2025-01-01, 2026-08-23] : {len(hors_plage_dates)}')

In [ ]:
# Q06 — Présentation des 10 cas argumentés
df_cas = pd.DataFrame(cas_hors_plage)
print(f'=== {len(df_cas)} cas hors contrat identifiés ===')
display(df_cas.head(10))

**Commentaire Q06 :** On vérifie trois types de problèmes :
1. **Valeurs hors plage** du dictionnaire (ex : `volume_m3 = 40123` dépasse 5000, probablement un point décimal manquant `40.123`)
2. **Contrainte métier** violée (`interruption_h > ouverture_h`)
3. **Dates impossibles** (mois 13, 31 février, etc.)

Un volume élevé seul ne constitue pas une fuite — c'est un usage intensif possible.

## Question 07 — Justifier chaque traitement

In [ ]:
# Q07 — Journal de préparation
journal = []

# ---- Règle 1 : Dédoublonner les lignes strictement identiques ----
n_before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
n_removed = n_before - len(df)
journal.append({'regle': 'Dé-doublonnage strict', 'colonnes': 'toutes',
                'effectif_touche': n_removed, 'justification': 'Lignes strictement identiques'})

# ---- Règle 2 : Harmoniser commune ----
commune_avant = df['commune'].copy()
df['commune'] = df['commune'].str.strip().str.title()
df['commune'] = df['commune'].str.replace(r'Zone_(\w)', lambda m: f'Zone_{m.group(1).upper()}', regex=True)
n_commune_fix = (commune_avant != df['commune']).sum()
journal.append({'regle': 'Harmonisation commune', 'colonnes': 'commune',
                'effectif_touche': int(n_commune_fix), 'justification': 'Espaces, casse hétérogènes'})

# ---- Règle 3 : Valeurs hors plage → NaN ----
n_invalide = 0
for col, (vmin, vmax) in plages.items():
    mask = df[col].notna() & ((df[col] < vmin) | (df[col] > vmax))
    n = mask.sum()
    n_invalide += n
    df.loc[mask, col] = np.nan
journal.append({'regle': 'Hors plage technique → NaN', 'colonnes': 'toutes numériques',
                'effectif_touche': n_invalide, 'justification': 'Valeurs dépassant les plages du dictionnaire'})

# ---- Règle 4 : Contrainte interruption <= ouverture → NaN l'interruption ----
mask_c = (df['interruption_h'].notna() & df['ouverture_h'].notna() &
          (df['interruption_h'] > df['ouverture_h']))
n_c = mask_c.sum()
df.loc[mask_c, 'interruption_h'] = np.nan
journal.append({'regle': 'Contrainte interruption > ouverture → NaN', 'colonnes': 'interruption_h',
                'effectif_touche': int(n_c), 'justification': 'Violation de la contrainte métier'})

# ---- Règle 5 : Imputation par la médiane (choix principal) ----
n_imputed = 0
for col in numeric_cols:
    mask_na = df[col].isna()
    n_na = mask_na.sum()
    if n_na > 0:
        median_val = df[col].median()
        df.loc[mask_na, col] = median_val
        n_imputed += n_na
journal.append({'regle': 'Imputation médiane', 'colonnes': 'toutes numériques',
                'effectif_touche': n_imputed,
                'justification': 'Médiane robuste aux valeurs extrêmes, conserve la distribution'})

# ---- Règle 6 : Suppression des lignes trop incomplètes (>50% NaN avant imputation) ----
journal.append({'regle': 'Lignes >50% NaN', 'colonnes': 'toutes numériques',
                'effectif_touche': 0,
                'justification': 'Aucune ligne n\'avait plus de 50% de manquants'})

df_journal = pd.DataFrame(journal)
print('=== Journal de préparation ===')
display(df_journal)

In [ ]:
# Q07 — Comparaison de deux stratégies d'imputation
print('=== Comparaison : médiane vs moyenne ===')

# Recréer un jeu avec NaN pour comparer
df_test = df_brut.copy()
for col in numeric_cols:
    df_test[col] = clean_numeric(df_test[col], col + '_test')

comparison = []
for col in numeric_cols:
    vals = df_test[col]
    comparison.append({
        'colonne': col,
        'NaN_count': vals.isna().sum(),
        'moyenne': round(vals.mean(), 3) if vals.notna().any() else None,
        'mediane': round(vals.median(), 3) if vals.notna().any() else None,
        'ecart': round(abs(vals.mean() - vals.median()), 3) if vals.notna().any() else None
    })
df_comparison = pd.DataFrame(comparison)
display(df_comparison)
print('\nChoix retenu : MÉDIANE — plus robuste aux valeurs extrêmes.')
print('La moyenne est sensible aux outliers (ex : volume_m3 avec des valeurs très élevées).')

**Commentaire Q07 :** On compare deux stratégies : imputation par la **médiane** (retenue) et par la **moyenne**. La médiane est plus robuste aux valeurs extrêmes, ce qui est important vu la présence de valeurs aberrantes dans les données. Chaque règle est documentée dans le journal avec sa justification et le nombre de valeurs touchées.

## Question 08 — Construire une référence descriptive

In [ ]:
# Q08 — Statistiques descriptives après nettoyage
print(f'=== Référence descriptive après nettoyage ===')
print(f'Observations restantes : {len(df)}')
print(f'Variables numériques : {len(numeric_cols)}')
print(f'Communes : {df["commune"].nunique()} → {sorted(df["commune"].unique())}')
print()
display(df[numeric_cols].describe().round(3))

In [ ]:
# Q08 — Distributions des variables numériques
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()
for i, col in enumerate(numeric_cols):
    axes[i].hist(df[col].dropna(), bins=30, color='steelblue', edgecolor='white', alpha=0.8)
    axes[i].set_title(col, fontsize=12, fontweight='bold')
    axes[i].set_ylabel('Fréquence')
for j in range(len(numeric_cols), len(axes)):
    axes[j].set_visible(False)
plt.suptitle('Distributions après nettoyage', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Q08 — Répartition par commune
fig, ax = plt.subplots(figsize=(8, 5))
df['commune'].value_counts().sort_index().plot(kind='bar', color='steelblue', edgecolor='white', ax=ax)
ax.set_title('Répartition des observations par commune', fontweight='bold')
ax.set_ylabel('Nombre d\'observations')
ax.set_xlabel('Commune')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Q08 — Matrice de corrélation
fig, ax = plt.subplots(figsize=(10, 8))
corr = df[numeric_cols].corr()
mask_tri = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask_tri, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax, square=True)
ax.set_title('Matrice de corrélation des variables numériques', fontweight='bold')
plt.tight_layout()
plt.show()

**Commentaire Q08 :** Après nettoyage, on dispose de l'ensemble des observations avec 7 variables numériques candidates. La variable `commune` sera conservée uniquement pour l'interprétation (ventilation des groupes par zone), sans être incluse dans les distances. Les distributions et la matrice de corrélation servent de référence pour interpréter les groupes.

---
# Phase 2 — Expérimentation
---

## Question 09 — Définir les expériences comparables

In [ ]:
# Q09 — Construction de la matrice numérique standardisée
features = ['volume_m3', 'passages', 'attente_min', 'ouverture_h',
            'pression_bar', 'interruption_h', 'stockage_m3']

X = df[features].values.copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f'Matrice numérique : {X_scaled.shape[0]} observations × {X_scaled.shape[1]} variables')
print(f'\nMoyennes après standardisation (≈0) : {X_scaled.mean(axis=0).round(6)}')
print(f'Écarts-types après standardisation (≈1) : {X_scaled.std(axis=0).round(6)}')

In [ ]:
# Q09 — Tableau des expériences planifiées
experiments = []
for k in range(2, 9):
    for seed in SEEDS:
        experiments.append({
            'representation': 'Standardisée (7 var.)',
            'methode': 'K-Means',
            'K': k,
            'n_init': 10,
            'graine': seed,
            'perimetre': f'{X_scaled.shape[0]} obs.'
        })
df_exp = pd.DataFrame(experiments)
print(f'=== Tableau des expériences : {len(df_exp)} configurations ===')
display(df_exp.head(10))
print(f'... ({len(df_exp)} lignes au total)')

## Question 10 — Exécuter une comparaison contrôlée

In [ ]:
# Q10 — K-Means pour K = 2 à 8, avec 5 graines
K_range = range(2, 9)
results = []

for k in K_range:
    for seed in SEEDS:
        km = KMeans(n_clusters=k, n_init=10, random_state=seed)
        labels = km.fit_predict(X_scaled)
        inertia = km.inertia_
        sil = silhouette_score(X_scaled, labels)
        sizes = np.bincount(labels).tolist()
        results.append({
            'K': k, 'graine': seed,
            'inertie': round(inertia, 2),
            'silhouette': round(sil, 4),
            'tailles': sizes
        })

df_results = pd.DataFrame(results)
print('=== Résultats K-Means ===')
display(df_results.head(15))

In [ ]:
# Q10 — Moyennes par K
summary = df_results.groupby('K').agg(
    inertie_moy=('inertie', 'mean'),
    inertie_std=('inertie', 'std'),
    silhouette_moy=('silhouette', 'mean'),
    silhouette_std=('silhouette', 'std')
).round(4)
display(summary)

In [ ]:
# Q10 — Courbe du coude (inertie) et silhouette
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Inertie
ax1.errorbar(summary.index, summary['inertie_moy'], yerr=summary['inertie_std'],
             marker='o', color='steelblue', capsize=4, linewidth=2)
ax1.set_xlabel('Nombre de groupes (K)', fontsize=12)
ax1.set_ylabel('Inertie moyenne', fontsize=12)
ax1.set_title('Courbe du coude', fontsize=13, fontweight='bold')
ax1.set_xticks(list(K_range))

# Silhouette
ax2.errorbar(summary.index, summary['silhouette_moy'], yerr=summary['silhouette_std'],
             marker='s', color='darkorange', capsize=4, linewidth=2)
ax2.set_xlabel('Nombre de groupes (K)', fontsize=12)
ax2.set_ylabel('Silhouette moyenne', fontsize=12)
ax2.set_title('Score de silhouette', fontsize=13, fontweight='bold')
ax2.set_xticks(list(K_range))

plt.tight_layout()
plt.show()

# Choix de K
best_k_sil = summary['silhouette_moy'].idxmax()
print(f'\nMeilleure silhouette moyenne pour K = {best_k_sil}')
print('Choix retenu : on examine K=3 et K=4 pour combiner silhouette et intérêt descriptif.')

In [ ]:
# Q10 — ACP pour visualisation 2D
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)

print(f'Variance expliquée par les 2 premières composantes : {pca.explained_variance_ratio_.sum():.2%}')
print(f'  PC1 : {pca.explained_variance_ratio_[0]:.2%}')
print(f'  PC2 : {pca.explained_variance_ratio_[1]:.2%}')

In [ ]:
# Q10 — Visualisation ACP colorée par cluster (K=3 et K=4)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, k in zip(axes, [3, 4]):
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    labels = km.fit_predict(X_scaled)
    
    scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=labels, cmap='Set2',
                        alpha=0.6, s=30, edgecolors='grey', linewidth=0.3)
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})', fontsize=11)
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})', fontsize=11)
    ax.set_title(f'K-Means K={k} (silhouette={silhouette_score(X_scaled, labels):.3f})',
                 fontsize=12, fontweight='bold')
    ax.legend(*scatter.legend_elements(), title='Groupe')

plt.suptitle('Projection ACP colorée par cluster K-Means', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Q10 — Choix final et modèle de référence
K_REF = 3
km_ref = KMeans(n_clusters=K_REF, n_init=10, random_state=RANDOM_STATE)
df['cluster'] = km_ref.fit_predict(X_scaled)

print(f'=== Partition de référence : K = {K_REF} ===')
print(f'Silhouette : {silhouette_score(X_scaled, df["cluster"]):.4f}')
print(f'\nTailles des groupes :')
print(df['cluster'].value_counts().sort_index())

# Profils (centres dans les unités originales)
print(f'\nProfils des groupes (moyennes dans les unités d\'origine) :')
profiles = df.groupby('cluster')[features].mean().round(2)
display(profiles)

## Question 11 — Étude spécifique 1 : Volume moyen par passage

In [ ]:
# Q11 — Volume moyen par passage (gestion passages = 0)
df['vol_par_passage'] = np.where(
    df['passages'] > 0,
    df['volume_m3'] / df['passages'],
    np.nan  # NaN si passages = 0 (éviter division par zéro)
)

n_zero_passages = (df['passages'] == 0).sum()
print(f'Bornes avec 0 passages : {n_zero_passages} → vol_par_passage = NaN')
print(f'\nStatistiques du volume par passage :')
display(df.groupby('cluster')['vol_par_passage'].describe().round(4))

In [ ]:
# Q11 — Boxplot par cluster
fig, ax = plt.subplots(figsize=(10, 6))
df.boxplot(column='vol_par_passage', by='cluster', ax=ax,
           patch_artist=True, boxprops=dict(facecolor='steelblue', alpha=0.5))
ax.set_title('Volume moyen par passage selon le cluster', fontsize=13, fontweight='bold')
ax.set_xlabel('Cluster')
ax.set_ylabel('Volume par passage (m³/passage)')
plt.suptitle('')
plt.tight_layout()
plt.show()

**Commentaire Q11 :** Les bornes avec 0 passages se voient attribuer `NaN` pour le ratio volume/passage (division invalide). Ce ratio permet de distinguer des bornes à forte consommation unitaire (peu de passages mais gros volumes) des bornes à usage fréquent mais faible par passage.

## Question 12 — Étude spécifique 2 : Volumes totaux vs intensités normalisées

In [ ]:
# Q12 — Segmentation sur volumes totaux vs intensité par heure ouverte

# Segmentation A : volumes totaux (toutes variables)
km_A = KMeans(n_clusters=K_REF, n_init=10, random_state=RANDOM_STATE)
labels_A = km_A.fit_predict(X_scaled)

# Intensité : volume / heure ouverte
df['intensite_vol_h'] = np.where(
    df['ouverture_h'] > 0,
    df['volume_m3'] / df['ouverture_h'],
    np.nan
)
df['intensite_pass_h'] = np.where(
    df['ouverture_h'] > 0,
    df['passages'] / df['ouverture_h'],
    np.nan
)

# Segmentation B : intensités uniquement
features_intensite = ['intensite_vol_h', 'intensite_pass_h', 'attente_min',
                      'pression_bar', 'interruption_h', 'stockage_m3']
X_int = df[features_intensite].dropna()
idx_int = X_int.index
scaler_int = StandardScaler()
X_int_scaled = scaler_int.fit_transform(X_int)

km_B = KMeans(n_clusters=K_REF, n_init=10, random_state=RANDOM_STATE)
labels_B_full = np.full(len(df), -1)
labels_B_full[idx_int] = km_B.fit_predict(X_int_scaled)

# Comparaison ARI
mask_common = labels_B_full >= 0
ari = adjusted_rand_score(labels_A[mask_common], labels_B_full[mask_common])
print(f'ARI entre segmentation volumes totaux et intensités : {ari:.4f}')
print(f'\nInterprétation : ARI de {ari:.2f} — les deux partitions sont ',
      'similaires.' if ari > 0.5 else 'assez différentes.')

In [ ]:
# Q12 — Visualisation côte à côte
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

ax1.scatter(X_pca[:, 0], X_pca[:, 1], c=labels_A, cmap='Set2', alpha=0.6, s=30, edgecolors='grey', linewidth=0.3)
ax1.set_title(f'Segmentation A — Volumes totaux\n(7 variables brutes)', fontweight='bold')
ax1.set_xlabel('PC1'); ax1.set_ylabel('PC2')

ax2.scatter(X_pca[mask_common, 0], X_pca[mask_common, 1],
            c=labels_B_full[mask_common], cmap='Set2', alpha=0.6, s=30, edgecolors='grey', linewidth=0.3)
ax2.set_title(f'Segmentation B — Intensités normalisées\n(volume/h, passages/h)', fontweight='bold')
ax2.set_xlabel('PC1'); ax2.set_ylabel('PC2')

plt.suptitle(f'Comparaison des segmentations (ARI = {ari:.3f})', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

**Commentaire Q12 :** La comparaison entre une segmentation fondée sur les volumes totaux et une autre fondée sur les intensités par heure ouverte montre si les mêmes bornes restent proches. L'ARI quantifie l'accord entre les deux partitions : un score proche de 1 indiquerait des partitions quasi identiques.

## Question 13 — Étude spécifique 3 : Noms descriptifs des groupes

In [ ]:
# Q13 — Description détaillée de chaque groupe
print('=== Profils des groupes dans les unités originales ===')
profile_detail = df.groupby('cluster')[features].agg(['mean', 'median', 'std']).round(2)
display(profile_detail)

# Tailles
print('\nTailles :')
print(df['cluster'].value_counts().sort_index())

In [ ]:
# Q13 — Graphique en coordonnées parallèles (implémentation manuelle)

# Normaliser min-max pour la visualisation
df_viz = df[features + ['cluster']].copy()
for col in features:
    cmin, cmax = df_viz[col].min(), df_viz[col].max()
    if cmax > cmin:
        df_viz[col] = (df_viz[col] - cmin) / (cmax - cmin)

# Moyennes par cluster
df_means = df_viz.groupby('cluster')[features].mean()

fig, ax = plt.subplots(figsize=(12, 6))
colors_map = ['#2ecc71', '#e74c3c', '#3498db', '#f39c12']
x = range(len(features))

for c in sorted(df_means.index):
    vals = df_means.loc[c, features].values
    color = colors_map[c % len(colors_map)]
    ax.plot(x, vals, marker='o', linewidth=3, label=f'Cluster {c}', color=color)

ax.set_xticks(x)
ax.set_xticklabels(features, rotation=30, ha='right')
ax.set_title('Profils des groupes (coordonnées parallèles, min-max normalisé)', fontweight='bold', fontsize=13)
ax.set_ylabel('Valeur normalisée [0-1]')
ax.legend(title='Cluster', fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Q13 — Attribution de noms descriptifs
profiles_mean = df.groupby('cluster')[features].mean()

noms_groupes = {}
for c in profiles_mean.index:
    p = profiles_mean.loc[c]
    descriptors = []
    if p['volume_m3'] > profiles_mean['volume_m3'].median():
        descriptors.append('Fort volume')
    else:
        descriptors.append('Volume modéré')
    if p['attente_min'] > profiles_mean['attente_min'].median():
        descriptors.append('forte attente')
    else:
        descriptors.append('faible attente')
    if p['interruption_h'] > profiles_mean['interruption_h'].median():
        descriptors.append('interruptions fréquentes')
    noms_groupes[c] = ' / '.join(descriptors)

print('=== Noms descriptifs des groupes ===')
for c, nom in noms_groupes.items():
    print(f'  Cluster {c} : "{nom}"')
print('\nNote : ces noms sont descriptifs et non normatifs (pas de jugement de qualité).')

## Question 14 — Étude spécifique 4 : Rôle de stockage_m3

In [ ]:
# Q14 — Comparer clustering avec et sans stockage_m3
features_sans_stockage = [f for f in features if f != 'stockage_m3']

X_sans_stock = df[features_sans_stockage].values
scaler_ss = StandardScaler()
X_sans_stock_scaled = scaler_ss.fit_transform(X_sans_stock)

km_avec = KMeans(n_clusters=K_REF, n_init=10, random_state=RANDOM_STATE)
labels_avec = km_avec.fit_predict(X_scaled)

km_sans = KMeans(n_clusters=K_REF, n_init=10, random_state=RANDOM_STATE)
labels_sans = km_sans.fit_predict(X_sans_stock_scaled)

ari_stockage = adjusted_rand_score(labels_avec, labels_sans)
print(f'ARI avec vs sans stockage_m3 : {ari_stockage:.4f}')
print(f'\nSilhouette avec stockage : {silhouette_score(X_scaled, labels_avec):.4f}')
print(f'Silhouette sans stockage : {silhouette_score(X_sans_stock_scaled, labels_sans):.4f}')

# Observations qui changent de groupe
changed = (labels_avec != labels_sans).sum()
print(f'\nObservations changeant de groupe : {changed} ({changed/len(df)*100:.1f}%)')

print(f'\nConclusion : ', end='')
if ari_stockage > 0.8:
    print('stockage_m3 a peu d\'influence → à utiliser pour l\'interprétation uniquement.')
else:
    print('stockage_m3 influence significativement les groupes → son inclusion modifie la partition.')

## Question 15 — Étude spécifique 5 : Retrait de pression_bar

In [ ]:
# Q15 — Retrait de pression_bar
features_sans_pression = [f for f in features if f != 'pression_bar']

X_sans_pres = df[features_sans_pression].values
scaler_sp = StandardScaler()
X_sans_pres_scaled = scaler_sp.fit_transform(X_sans_pres)

km_sans_pres = KMeans(n_clusters=K_REF, n_init=10, random_state=RANDOM_STATE)
labels_sans_pres = km_sans_pres.fit_predict(X_sans_pres_scaled)

ari_pression = adjusted_rand_score(labels_avec, labels_sans_pres)
print(f'ARI avec vs sans pression_bar : {ari_pression:.4f}')

# Identification des bornes qui changent
mask_change = labels_avec != labels_sans_pres
n_change = mask_change.sum()
print(f'Bornes changeant de groupe : {n_change} ({n_change/len(df)*100:.1f}%)')

if n_change > 0:
    print(f'\nExemples de bornes affectées :')
    df_changed = df[mask_change][['id_observation', 'commune'] + features].copy()
    df_changed['cluster_avec'] = labels_avec[mask_change]
    df_changed['cluster_sans'] = labels_sans_pres[mask_change]
    display(df_changed.head(10))

In [ ]:
# Q15 — Visualisation des changements
fig, ax = plt.subplots(figsize=(10, 6))
colors = np.where(mask_change, 'red', 'lightgrey')
ax.scatter(X_pca[:, 0], X_pca[:, 1], c=colors, alpha=0.6, s=30, edgecolors='grey', linewidth=0.3)
ax.scatter([], [], c='red', label=f'Changent de groupe (n={n_change})')
ax.scatter([], [], c='lightgrey', label=f'Stables (n={len(df)-n_change})')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.set_title('Bornes changeant de groupe après retrait de pression_bar', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

## Question 16 — Étude spécifique 6 : Visites techniques prioritaires

In [ ]:
# Q16 — Identification de 2 visites techniques prioritaires

# Critère 1 : bornes à forte attente et interruptions élevées
score_urgence = (
    (df['attente_min'] - df['attente_min'].mean()) / df['attente_min'].std() +
    (df['interruption_h'] - df['interruption_h'].mean()) / df['interruption_h'].std()
)
idx_visite1 = score_urgence.idxmax()
visite1 = df.loc[idx_visite1]

# Critère 2 : borne avec très fort volume et pression basse (risque technique)
score_risque = (
    (df['volume_m3'] - df['volume_m3'].mean()) / df['volume_m3'].std() -
    (df['pression_bar'] - df['pression_bar'].mean()) / df['pression_bar'].std()
)
# Exclure la première visite pour diversifier
score_risque.loc[idx_visite1] = -np.inf
idx_visite2 = score_risque.idxmax()
visite2 = df.loc[idx_visite2]

print('=== VISITE TECHNIQUE PRIORITAIRE 1 ===')
print(f'Borne : {visite1["id_observation"]} (Commune : {visite1["commune"]})')
print(f'Motif : Attente très élevée ({visite1["attente_min"]:.1f} min) et interruptions')
print(f'        importantes ({visite1["interruption_h"]:.1f} h)')
print(f'Cluster : {visite1["cluster"]}')
print(f'Info à recueillir : état physique de la borne, débit réel, file d\'attente observée')

print(f'\n=== VISITE TECHNIQUE PRIORITAIRE 2 ===')
print(f'Borne : {visite2["id_observation"]} (Commune : {visite2["commune"]})')
print(f'Motif : Volume très élevé ({visite2["volume_m3"]:.1f} m³) avec pression basse')
print(f'        ({visite2["pression_bar"]:.2f} bar) — possible fuite ou surcharge')
print(f'Cluster : {visite2["cluster"]}')
print(f'Info à recueillir : inspection des canalisations, mesure de pression en temps réel')

---
# Phase 3 — Évaluation et robustesse
---

## Question 17 — Évaluer les partitions

In [ ]:
# Q17 — Évaluation complète des partitions K=2 à 8
evaluation = []
for k in K_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    labels = km.fit_predict(X_scaled)
    sil = silhouette_score(X_scaled, labels)
    sizes = np.bincount(labels)
    evaluation.append({
        'K': k,
        'silhouette': round(sil, 4),
        'inertie': round(km.inertia_, 2),
        'taille_min': sizes.min(),
        'taille_max': sizes.max(),
        'ratio_max_min': round(sizes.max() / sizes.min(), 2),
        'tailles': sizes.tolist()
    })

df_eval = pd.DataFrame(evaluation)
print('=== Évaluation des partitions ===')
display(df_eval)

In [ ]:
# Q17 — Profils des centres dans les unités d'origine
print(f'\n=== Profils pour K = {K_REF} (unités d\'origine) ===')
profiles_orig = df.groupby('cluster')[features].mean().round(2)
display(profiles_orig)

# Heatmap des profils normalisés
fig, ax = plt.subplots(figsize=(12, 5))
profiles_norm = (profiles_orig - profiles_orig.mean()) / profiles_orig.std()
sns.heatmap(profiles_norm, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, ax=ax, linewidths=1)
ax.set_title(f'Profils des groupes normalisés (K={K_REF})', fontweight='bold', fontsize=13)
ax.set_ylabel('Cluster')
plt.tight_layout()
plt.show()

**Commentaire Q17 :** La silhouette exige au moins 2 groupes et moins de groupes que d'observations. On compare non seulement la silhouette mais aussi les tailles (un groupe minuscule est suspect), les profils (interprétabilité) et l'intérêt descriptif pour la régie. Une bonne silhouette ne prouve pas l'existence de catégories naturelles.

## Question 18 — Tester la stabilité des groupes

In [ ]:
# Q18 — Stabilité inter-graines
print(f'=== Test de stabilité pour K = {K_REF} ===')

all_labels = {}
for seed in SEEDS:
    km = KMeans(n_clusters=K_REF, n_init=10, random_state=seed)
    all_labels[seed] = km.fit_predict(X_scaled)

# Matrice ARI entre paires de graines
ari_matrix = pd.DataFrame(index=SEEDS, columns=SEEDS, dtype=float)
for s1 in SEEDS:
    for s2 in SEEDS:
        ari_matrix.loc[s1, s2] = adjusted_rand_score(all_labels[s1], all_labels[s2])

print('Matrice ARI entre graines :')
display(ari_matrix.round(4))

# ARI moyen hors diagonale
ari_vals = []
for i, s1 in enumerate(SEEDS):
    for j, s2 in enumerate(SEEDS):
        if i < j:
            ari_vals.append(ari_matrix.loc[s1, s2])
print(f'\nARI moyen entre paires : {np.mean(ari_vals):.4f} (±{np.std(ari_vals):.4f})')
print('Un ARI > 0.9 indique une partition très stable.')

In [ ]:
# Q18 — Identifier les observations instables
labels_array = np.array([all_labels[s] for s in SEEDS])  # shape (5, n)

# Pour chaque observation, compter combien de fois elle change de groupe relatif
stability_count = np.zeros(len(df))
ref_labels = labels_array[0]  # graine de référence
for i in range(1, len(SEEDS)):
    ari_i = adjusted_rand_score(ref_labels, labels_array[i])
    # Observation-level: est-ce que l'observation garde le même voisinage?
    for obs_idx in range(len(df)):
        ref_cluster = ref_labels[obs_idx]
        ref_members = set(np.where(ref_labels == ref_cluster)[0])
        new_cluster = labels_array[i][obs_idx]
        new_members = set(np.where(labels_array[i] == new_cluster)[0])
        overlap = len(ref_members & new_members) / max(len(ref_members | new_members), 1)
        stability_count[obs_idx] += overlap

stability_score = stability_count / (len(SEEDS) - 1)
df['stabilite'] = stability_score

n_instable = (stability_score < 0.5).sum()
print(f'Observations très instables (score < 0.5) : {n_instable}')
print(f'Observations stables (score > 0.8) : {(stability_score > 0.8).sum()}')

## Question 19 — Mesurer l'influence du nettoyage

In [ ]:
# Q19 — Alternative : imputation par la MOYENNE au lieu de la médiane
df_alt = df_brut.copy()
conversion_log_alt = []
for col in numeric_cols:
    df_alt[col] = clean_numeric(df_alt[col], col + '_alt')

# Appliquer le même dé-doublonnage que pour df
df_alt = df_alt.drop_duplicates().reset_index(drop=True)

# Harmoniser commune
df_alt['commune'] = df_alt['commune'].str.strip().str.title()
df_alt['commune'] = df_alt['commune'].str.replace(r'Zone_(\w)', lambda m: f'Zone_{m.group(1).upper()}', regex=True)

# Hors plage → NaN
for col, (vmin, vmax) in plages.items():
    mask = df_alt[col].notna() & ((df_alt[col] < vmin) | (df_alt[col] > vmax))
    df_alt.loc[mask, col] = np.nan

# Contrainte métier
mask_c_alt = (df_alt['interruption_h'].notna() & df_alt['ouverture_h'].notna() &
              (df_alt['interruption_h'] > df_alt['ouverture_h']))
df_alt.loc[mask_c_alt, 'interruption_h'] = np.nan

# Imputation par la MOYENNE (alternative)
for col in numeric_cols:
    mask_na = df_alt[col].isna()
    if mask_na.sum() > 0:
        df_alt.loc[mask_na, col] = df_alt[col].mean()

# Clustering avec les mêmes paramètres
X_alt = df_alt[features].values
scaler_alt = StandardScaler()
X_alt_scaled = scaler_alt.fit_transform(X_alt)

km_alt = KMeans(n_clusters=K_REF, n_init=10, random_state=RANDOM_STATE)
labels_alt = km_alt.fit_predict(X_alt_scaled)

# Comparaison
ari_nettoyage = adjusted_rand_score(df['cluster'].values, labels_alt)
sil_median = silhouette_score(X_scaled, df['cluster'].values)
sil_mean = silhouette_score(X_alt_scaled, labels_alt)

print('=== Impact du choix d\'imputation ===')
print(f'ARI médiane vs moyenne : {ari_nettoyage:.4f}')
print(f'Silhouette (imputation médiane) : {sil_median:.4f}')
print(f'Silhouette (imputation moyenne) : {sil_mean:.4f}')

# Changements
changed_nettoyage = (df['cluster'].values != labels_alt).sum()
print(f'\nObservations changeant de groupe : {changed_nettoyage} ({changed_nettoyage/len(df)*100:.1f}%)')
print(f'\nConclusion : la conclusion dépend ',
      'davantage de l\'algorithme.' if ari_nettoyage > 0.8 else 'significativement de la décision de préparation.')

## Question 20 — Documenter les cas difficiles

In [ ]:
# Q20 — 5 cas ambigus, isolés ou mal représentés
sil_samples = silhouette_samples(X_scaled, df['cluster'].values)
df['silhouette_indiv'] = sil_samples

# Les 5 observations avec la plus faible silhouette (les plus ambiguës)
cas_difficiles = df.nsmallest(5, 'silhouette_indiv')

print('=== 5 cas difficiles (silhouette individuelle la plus basse) ===\n')
for i, (idx, row) in enumerate(cas_difficiles.iterrows(), 1):
    print(f'--- Cas {i} : {row["id_observation"]} ---')
    print(f'  Commune : {row["commune"]}')
    print(f'  Cluster assigné : {row["cluster"]} | Silhouette : {row["silhouette_indiv"]:.4f}')
    print(f'  Valeurs : volume={row["volume_m3"]:.1f} m³, passages={row["passages"]:.0f}, '
          f'attente={row["attente_min"]:.1f} min')
    print(f'            ouverture={row["ouverture_h"]:.1f} h, pression={row["pression_bar"]:.2f} bar, '
          f'interruption={row["interruption_h"]:.1f} h, stockage={row["stockage_m3"]:.1f} m³')
    
    # Deux interprétations
    current_cluster = int(row['cluster'])
    distances = np.linalg.norm(X_scaled[idx] - km_ref.cluster_centers_, axis=1)
    sorted_clusters = np.argsort(distances)
    second_cluster = sorted_clusters[1] if sorted_clusters[0] == current_cluster else sorted_clusters[0]
    
    print(f'  Interprétation 1 : appartient au groupe {current_cluster} '
          f'(distance au centre = {distances[current_cluster]:.2f})')
    print(f'  Interprétation 2 : pourrait appartenir au groupe {second_cluster} '
          f'(distance au centre = {distances[second_cluster]:.2f})')
    print(f'  Info nécessaire : visite terrain pour confirmer le profil de fonctionnement.\n')

---
# Phase 4 — Interprétation et reproductibilité
---

## Question 21 — Relier résultats et décision

In [ ]:
# Q21 — Synthèse de la typologie
print('=' * 70)
print('RÉPONSE À LA MISSION')
print('Construire une typologie descriptive des bornes et distinguer')
print('intensité de service, disponibilité et attente.')
print('=' * 70)

print(f'\nLa partition en K={K_REF} groupes, obtenue par K-Means sur les 7 variables')
print('standardisées, permet de distinguer les profils suivants :\n')

for c in sorted(df['cluster'].unique()):
    p = profiles_orig.loc[c]
    n = (df['cluster'] == c).sum()
    print(f'  Groupe {c} ({noms_groupes.get(c, "")}) — {n} bornes')
    print(f'    Volume moyen : {p["volume_m3"]:.1f} m³ | Passages : {p["passages"]:.0f}')
    print(f'    Attente : {p["attente_min"]:.1f} min | Ouverture : {p["ouverture_h"]:.1f} h')
    print(f'    Interruption : {p["interruption_h"]:.1f} h | Pression : {p["pression_bar"]:.2f} bar')
    print()

print('\n=== Actions exploratoires proposées ===')
print('\n1. Vérification terrain des bornes à forte attente et interruptions :')
print('   → Résultat : un groupe se distingue par des attentes élevées.')
print('   → Limite : les données sont fictives, la corrélation attente-interruption')
print('     pourrait ne pas refléter une relation causale réelle.')

print('\n2. Collecte de données supplémentaires sur l\'état technique :')
print('   → Résultat : la pression seule ne discrimine pas bien les groupes.')
print('   → Limite : sans diagnostic technique réel, on ne peut pas distinguer')
print('     un usage intensif normal d\'un dysfonctionnement.')

## Question 22 — Présenter les limites

In [ ]:
# Q22 — Limites
print('=== LIMITES DU TRAVAIL ===')

print('\n1. LIMITE DE TERRAIN (données fictives) :')
print('   Les données sont entièrement simulées. Les profils identifiés ne correspondent')
print('   à aucune réalité observée. Les plages techniques du dictionnaire ne constituent')
print('   pas des normes réelles. Avant toute utilisation opérationnelle, il faudrait')
print('   valider la typologie sur des données réelles.')

print('\n2. LIMITE STATISTIQUE (hypothèses K-Means) :')
print('   K-Means suppose des groupes convexes, de taille comparable et de variance')
print('   homogène. L\'imputation par la médiane biaise les distributions vers le')
print('   centre. Le choix de K reste subjectif malgré les métriques. La standardisation')
print('   donne un poids égal à toutes les variables, ce qui n\'est pas nécessairement')
print('   justifié du point de vue métier.')

print('\n3. LIMITE DE GÉNÉRALISATION :')
print('   La typologie est spécifique aux 4 communes et à la période observée.')
print('   Elle ne peut pas être extrapolée à d\'autres territoires, saisons ou')
print('   infrastructures sans validation. Les seuils de groupe ne constituent pas')
print('   des règles de décision automatique.')

## Question 23 — Rendre l'analyse reproductible

In [ ]:
# Q23 — Reproductibilité
import sklearn, sys
print('=== INFORMATIONS DE REPRODUCTIBILITÉ ===')
print(f'Python : {sys.version}')
print(f'NumPy : {np.__version__}')
print(f'Pandas : {pd.__version__}')
print(f'Scikit-learn : {sklearn.__version__}')
print(f'Matplotlib : {plt.matplotlib.__version__}')
print(f'Seaborn : {sns.__version__}')
print(f'\nGraine aléatoire principale : RANDOM_STATE = {RANDOM_STATE}')
print(f'Graines de stabilité : SEEDS = {SEEDS}')
print(f'Fichier de données : chemin relatif via glob')
print(f'\nPour reproduire : redémarrer le noyau et exécuter toutes les cellules.')
print('Aucune modification manuelle des CSV bruts n\'est nécessaire.')

## Question 24 — Défendre la conclusion

In [ ]:
# Q24 — Éléments pour la restitution de 10 minutes
print('=' * 70)
print('RESTITUTION — Profils de fonctionnement des bornes fontaines')
print('=' * 70)

print(f'''
RÉSULTAT PRINCIPAL :
  K-Means avec K={K_REF} groupes identifie des profils distincts de
  fonctionnement des bornes fontaines, caractérisés par l'intensité
  de service (volume, passages), la disponibilité (ouverture,
  interruptions) et l'attente.

COMPARAISON QUI LE SOUTIENT :
  La segmentation est stable (ARI moyen = {np.mean(ari_vals):.3f} entre graines)
  et le choix d'imputation (médiane vs moyenne) influence modérément
  les résultats (ARI = {ari_nettoyage:.3f}).

CAS DIFFICILE :
  {cas_difficiles.iloc[0]['id_observation']} présente une silhouette de
  {cas_difficiles.iloc[0]['silhouette_indiv']:.3f}, à la frontière entre deux groupes.
  Une visite terrain serait nécessaire pour confirmer son profil.

LIMITE DÉCISIVE :
  Les données sont fictives. La typologie ne peut pas être utilisée
  pour des décisions opérationnelles sans validation sur données réelles.

CE QUE LE TRAVAIL PERMET D'EXPLORER :
  - Des profils de fonctionnement à investiguer sur le terrain
  - Des priorités de visite technique basées sur les caractéristiques

CE QU'IL FAUDRAIT ENCORE VÉRIFIER :
  - La stabilité temporelle des profils (données sur plusieurs mois)
  - L'ajout de variables terrain (état physique, ancienneté, quartier)
''')

In [ ]:
# Sauvegarde du journal de préparation
df_journal.to_csv('journal_preparation.csv', index=False, encoding='utf-8-sig')

# Sauvegarde du tableau des expériences
df_results.to_csv('tableau_experiences.csv', index=False, encoding='utf-8-sig')

# Sauvegarde des données nettoyées
df.to_csv('donnees_nettoyees.csv', index=False, sep=';', encoding='utf-8-sig')

print('Fichiers sauvegardés :')
print('  - journal_preparation.csv')
print('  - tableau_experiences.csv')
print('  - donnees_nettoyees.csv')
print('\n=== FIN DU NOTEBOOK ===')